# Counterfactual Risk Sensitivity Atlas (Colab)


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/notebooks/counterfactual_risk_sensitivity_atlas_colab.ipynb)


## Research Questions
1. Which perturbation factors most strongly increase high-risk outcomes?
2. Are top factors robust across outcome-threshold settings?
3. Are factor rankings method-consistent under a multi-objective rulebook?

### Repo-Inspired Components
- **VerifAI**: multi-objective/rulebook ranking and falsification-style prioritization.
- **SEAL**: realism-oriented consistency checks across methods.
- **STRIVE**: scenario-family analysis via threshold robustness.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/achyutmorang/waymax-simulation-experiments.git"
REPO_DIR = "/content/waymax-simulation-experiments"

if os.path.exists(REPO_DIR):
    print("Repo exists, fast-forwarding main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Mount Google Drive in Colab
import os

if os.environ.get('COLAB_RELEASE_TAG'):
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        else:
            print('[drive] /content/drive already mounted')
    except Exception as e:
        print('[drive] mount skipped:', e)
else:
    print('[drive] non-Colab environment; skipping mount')


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.eval_counterfactual_risk_sensitivity import (
    CounterfactualHypothesisConfig,
    aggregate_factor_importance,
    bootstrap_factor_importance_ci,
    build_sensitivity_atlas,
    compute_high_risk_event_flag,
    discover_and_load_trace,
    evaluate_counterfactual_hypotheses,
    factor_response_profile,
    load_intervention_tables,
    method_factor_importance,
    permutation_test_factor_nonzero,
    plot_factor_importance_ci,
    plot_factor_slope_distribution,
    plot_method_factor_heatmap,
    plot_response_profile,
    save_figure,
    top_sensitive_scenarios,
    trace_to_intervention_table,
)
from src.eval_compute_normalized_discovery import rulebook_lexicographic_ranking

pd.set_option('display.max_columns', 300)


In [ ]:
# ===== User config =====
RUN_TAG = "closedloop_v1"
SIM_PERSIST_ROOT = "/content/drive/MyDrive/waymax_experiments/closedloop_runs/v1"
N_SHARDS = 5
OUT_RUN_TAG = None
PREFER_KIND = "merged"

INTERVENTION_CSVS = []

FACTOR_COLUMNS = ("delta_x", "delta_y", "delta_l2", "step_scale")
MIN_UNIQUE_VALUES = 3
TOP_N_SCENARIOS = 30
BOOTSTRAP_SAMPLES = 2000
PERMUTATION_SAMPLES = 5000

OUTCOME_RISK_QUANTILES = [0.85, 0.90, 0.95]
OUTCOME_NEAR_MISS_TTC = 2.0

HYPOTHESIS_CONFIG = CounterfactualHypothesisConfig(
    alpha=0.05,
    min_mean_abs_slope=0.02,
    min_monotonic_fraction=0.50,
    min_top_factor_method_consistency=0.60,
    min_significant_factors=1,
)

# VerifAI-inspired factor rulebook
FACTOR_RULEBOOK_PRIORITIES = [
    "mean_abs_slope",          # maximize
    "monotonic_fraction",      # maximize
    "method_top3_consistency", # maximize
    "neg_log10_p",             # maximize
]
FACTOR_RULEBOOK_MAXIMIZE = {
    "mean_abs_slope": True,
    "monotonic_fraction": True,
    "method_top3_consistency": True,
    "neg_log10_p": True,
}

EXPORT_DIR = f"/content/drive/MyDrive/waymax_experiments/counterfactual_risk_sensitivity_atlas/{RUN_TAG}"


In [ ]:
run_data = None
if len(INTERVENTION_CSVS) > 0:
    print("[precheck] Using provided intervention CSVs. Skipping simulation artifact lookup.")
    intervention_raw_df = load_intervention_tables(INTERVENTION_CSVS)
    if {"factor_name", "factor_value"}.issubset(intervention_raw_df.columns):
        id_cols = [
            c
            for c in ["scenario_id", "method", "eval_index", "seed_used", "risk_sks", "failure_proxy", "surprise_pd"]
            if c in intervention_raw_df.columns
        ]
        trace_df = intervention_raw_df.pivot_table(
            index=id_cols,
            columns="factor_name",
            values="factor_value",
            aggfunc="mean",
        ).reset_index()
    else:
        trace_df = intervention_raw_df.copy()
else:
    try:
        run_data = discover_and_load_trace(
            run_tag=RUN_TAG,
            persist_root=SIM_PERSIST_ROOT,
            n_shards=N_SHARDS,
            out_run_tag=OUT_RUN_TAG,
            prefer_kind=PREFER_KIND,
        )
    except FileNotFoundError as e:
        print("[precheck] No completed simulation artifacts were found for this RUN_TAG/SIM_PERSIST_ROOT.")
        print("[precheck] Run closedloop_simulation_colab.ipynb first, or update RUN_TAG/SIM_PERSIST_ROOT.")
        raise
    except ValueError as e:
        print("[precheck] Artifact discovery failed because required files/columns are missing.")
        print("[precheck] Verify merged/shard exports include per_eval_trace with factor columns.")
        raise
    print(f"[precheck] Found simulation artifacts at: {run_data.selected_run_prefix}")
    print("[precheck] Skipping simulation and running evaluation/analysis only.")
    display(run_data.candidates_df)
    print("selected_run_prefix:", run_data.selected_run_prefix)
    trace_df = run_data.trace_df.copy()

print("trace rows:", len(trace_df))
display(trace_df.head())

long_df = trace_to_intervention_table(trace_df, factor_columns=FACTOR_COLUMNS)
print("long intervention rows:", len(long_df))
if len(long_df) > 0:
    display(long_df.head())


In [ ]:
# Outcome-threshold robustness sweep.
robust_rows = []
robust_factor_rows = []
for rq in OUTCOME_RISK_QUANTILES:
    work = trace_df.copy()
    high_flag, meta = compute_high_risk_event_flag(
        work,
        risk_quantile=float(rq),
        near_miss_ttc_threshold=float(OUTCOME_NEAR_MISS_TTC),
    )
    work["is_high_risk_event"] = high_flag.astype(int)

    atlas_df_q, _ = build_sensitivity_atlas(
        trace_df=work,
        factor_columns=FACTOR_COLUMNS,
        outcome_col="is_high_risk_event",
        min_unique_values=MIN_UNIQUE_VALUES,
    )
    imp_df_q = aggregate_factor_importance(atlas_df_q)

    top_factor = str(imp_df_q.iloc[0]["factor_name"]) if not imp_df_q.empty else ""
    top_strength = float(imp_df_q.iloc[0]["mean_abs_slope"]) if not imp_df_q.empty else np.nan
    robust_rows.append(
        {
            "risk_quantile": float(rq),
            "risk_threshold": float(meta.get("risk_threshold", np.nan)),
            "top_factor": top_factor,
            "top_mean_abs_slope": top_strength,
            "n_atlas_rows": int(len(atlas_df_q)),
        }
    )

    if not imp_df_q.empty:
        tmp = imp_df_q[["factor_name", "mean_abs_slope", "monotonic_fraction"]].copy()
        tmp["risk_quantile"] = float(rq)
        robust_factor_rows.append(tmp)

robust_summary_df = pd.DataFrame(robust_rows).sort_values("risk_quantile").reset_index(drop=True)
robust_factor_df = pd.concat(robust_factor_rows, ignore_index=True) if robust_factor_rows else pd.DataFrame()

print("Outcome-threshold robustness summary")
display(robust_summary_df)


In [ ]:
# Main threshold detailed analysis.
main_quantile = OUTCOME_RISK_QUANTILES[len(OUTCOME_RISK_QUANTILES) // 2]
work_main = trace_df.copy()
main_flag, main_meta = compute_high_risk_event_flag(
    work_main,
    risk_quantile=float(main_quantile),
    near_miss_ttc_threshold=float(OUTCOME_NEAR_MISS_TTC),
)
work_main["is_high_risk_event"] = main_flag.astype(int)

atlas_df, _ = build_sensitivity_atlas(
    trace_df=work_main,
    factor_columns=FACTOR_COLUMNS,
    outcome_col="is_high_risk_event",
    min_unique_values=MIN_UNIQUE_VALUES,
)
importance_df = aggregate_factor_importance(atlas_df)
method_importance_df = method_factor_importance(atlas_df)
top_scenarios_df = top_sensitive_scenarios(atlas_df, top_n=TOP_N_SCENARIOS)
ci_df = bootstrap_factor_importance_ci(atlas_df, n_boot=BOOTSTRAP_SAMPLES)

perm_rows = []
for factor_name in importance_df["factor_name"].tolist() if not importance_df.empty else []:
    perm_rows.append(
        permutation_test_factor_nonzero(
            atlas_df,
            factor_name=factor_name,
            n_perm=PERMUTATION_SAMPLES,
        )
    )
perm_df = pd.DataFrame(perm_rows)

hypothesis_df, hypothesis_artifacts = evaluate_counterfactual_hypotheses(
    atlas_df=atlas_df,
    importance_df=importance_df,
    permutation_df=perm_df,
    config=HYPOTHESIS_CONFIG,
)

# VerifAI-inspired factor rulebook ranking.
if not importance_df.empty:
    factor_table = importance_df.copy()
    if not perm_df.empty and {"factor_name", "p_value_two_sided"}.issubset(perm_df.columns):
        factor_table = factor_table.merge(
            perm_df[["factor_name", "p_value_two_sided"]],
            on="factor_name",
            how="left",
        )
    else:
        factor_table["p_value_two_sided"] = np.nan

    if not method_importance_df.empty:
        top3 = (
            method_importance_df.sort_values(["method", "mean_abs_slope"], ascending=[True, False])
            .groupby("method", as_index=False)
            .head(3)
        )
        n_methods = max(1, int(method_importance_df["method"].nunique()))
        consistency = (
            top3.groupby("factor_name", as_index=False)
            .agg(method_top3_consistency=("method", lambda x: float(x.nunique() / n_methods)))
        )
        factor_table = factor_table.merge(consistency, on="factor_name", how="left")
    else:
        factor_table["method_top3_consistency"] = np.nan

    factor_table["method_top3_consistency"] = factor_table["method_top3_consistency"].fillna(0.0)
    factor_table["neg_log10_p"] = -np.log10(
        pd.to_numeric(factor_table["p_value_two_sided"], errors="coerce").fillna(1.0).clip(lower=1e-12)
    )

    rank_input = factor_table.rename(columns={"factor_name": "method"})
    factor_rulebook_rank_df = rulebook_lexicographic_ranking(
        rank_input,
        priorities=FACTOR_RULEBOOK_PRIORITIES,
        maximize=FACTOR_RULEBOOK_MAXIMIZE,
    ).rename(columns={"method": "factor_name"})
else:
    factor_rulebook_rank_df = pd.DataFrame()

print("Main outcome metadata:", main_meta)
print("=== Hypothesis Verdicts ===")
display(hypothesis_df)
print("=== Factor Importance ===")
display(importance_df)
print("=== Method-wise Factor Importance ===")
display(method_importance_df)
print("=== Factor Rulebook Ranking ===")
display(factor_rulebook_rank_df)


In [ ]:
fig_imp = plot_factor_importance_ci(importance_df, ci_df, figsize=(8, 4))
if fig_imp is not None:
    plt.show()

fig_slope = plot_factor_slope_distribution(atlas_df, figsize=(9, 4))
if fig_slope is not None:
    plt.show()

fig_heat = plot_method_factor_heatmap(method_importance_df, value_col="mean_abs_slope", figsize=(10, 4.5))
if fig_heat is not None:
    plt.show()

if not robust_factor_df.empty:
    pivot = robust_factor_df.pivot_table(
        index="factor_name",
        columns="risk_quantile",
        values="mean_abs_slope",
        aggfunc="mean",
    )
    if not pivot.empty:
        plt.figure(figsize=(8, 4.5))
        plt.imshow(pivot.to_numpy(dtype=float), aspect="auto")
        plt.xticks(np.arange(len(pivot.columns)), [f"q={c:.2f}" for c in pivot.columns])
        plt.yticks(np.arange(len(pivot.index)), pivot.index)
        plt.colorbar(label="mean |slope|")
        plt.title("Factor Strength Robustness Across Outcome Thresholds")
        plt.tight_layout()
        plt.show()

for fac in importance_df["factor_name"].head(2).tolist() if not importance_df.empty else []:
    prof_df = factor_response_profile(
        trace_df=work_main,
        factor_col=str(fac),
        outcome_col="is_high_risk_event",
        method_col="method",
        n_bins=10,
    )
    fig_prof = plot_response_profile(prof_df, factor_name=str(fac), figsize=(8, 4))
    if fig_prof is not None:
        plt.show()


In [ ]:
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
plots_dir = Path(EXPORT_DIR) / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
run_prefix_name = Path(run_data.selected_run_prefix).name if run_data is not None else "intervention_csv"

out = {
    "robust_summary": f"{EXPORT_DIR}/{run_prefix_name}_outcome_robustness_summary.csv",
    "robust_factor": f"{EXPORT_DIR}/{run_prefix_name}_outcome_robustness_factors.csv",
    "atlas": f"{EXPORT_DIR}/{run_prefix_name}_atlas.csv",
    "importance": f"{EXPORT_DIR}/{run_prefix_name}_factor_importance.csv",
    "method_importance": f"{EXPORT_DIR}/{run_prefix_name}_method_factor_importance.csv",
    "top_scenarios": f"{EXPORT_DIR}/{run_prefix_name}_top_sensitive_scenarios.csv",
    "ci": f"{EXPORT_DIR}/{run_prefix_name}_factor_importance_ci.csv",
    "permutation": f"{EXPORT_DIR}/{run_prefix_name}_factor_permutation_tests.csv",
    "hypothesis": f"{EXPORT_DIR}/{run_prefix_name}_hypothesis_verdicts.csv",
    "factor_rulebook": f"{EXPORT_DIR}/{run_prefix_name}_factor_rulebook_ranking.csv",
    "outcome_meta": f"{EXPORT_DIR}/{run_prefix_name}_outcome_metadata.json",
}

robust_summary_df.to_csv(out["robust_summary"], index=False)
robust_factor_df.to_csv(out["robust_factor"], index=False)
atlas_df.to_csv(out["atlas"], index=False)
importance_df.to_csv(out["importance"], index=False)
method_importance_df.to_csv(out["method_importance"], index=False)
top_scenarios_df.to_csv(out["top_scenarios"], index=False)
ci_df.to_csv(out["ci"], index=False)
perm_df.to_csv(out["permutation"], index=False)
hypothesis_df.to_csv(out["hypothesis"], index=False)
factor_rulebook_rank_df.to_csv(out["factor_rulebook"], index=False)

with open(out["outcome_meta"], "w") as f:
    json.dump(
        {
            "run_prefix": run_data.selected_run_prefix if run_data is not None else None,
            "main_quantile": float(main_quantile),
            "main_outcome_metadata": main_meta,
            "rulebook_priorities": FACTOR_RULEBOOK_PRIORITIES,
            "hypothesis_config": {
                "alpha": HYPOTHESIS_CONFIG.alpha,
                "min_mean_abs_slope": HYPOTHESIS_CONFIG.min_mean_abs_slope,
                "min_monotonic_fraction": HYPOTHESIS_CONFIG.min_monotonic_fraction,
                "min_top_factor_method_consistency": HYPOTHESIS_CONFIG.min_top_factor_method_consistency,
                "min_significant_factors": HYPOTHESIS_CONFIG.min_significant_factors,
            },
        },
        f,
        indent=2,
    )

save_figure(plot_factor_importance_ci(importance_df, ci_df, figsize=(8, 4)), str(plots_dir / f"{run_prefix_name}_factor_importance_ci.png"))
save_figure(plot_factor_slope_distribution(atlas_df, figsize=(9, 4)), str(plots_dir / f"{run_prefix_name}_slope_distribution.png"))
save_figure(plot_method_factor_heatmap(method_importance_df, value_col="mean_abs_slope", figsize=(10, 4.5)), str(plots_dir / f"{run_prefix_name}_method_factor_heatmap.png"))
for fac in importance_df["factor_name"].head(2).tolist() if not importance_df.empty else []:
    prof_df = factor_response_profile(
        trace_df=work_main,
        factor_col=str(fac),
        outcome_col="is_high_risk_event",
        method_col="method",
        n_bins=10,
    )
    save_figure(plot_response_profile(prof_df, factor_name=str(fac), figsize=(8, 4)), str(plots_dir / f"{run_prefix_name}_response_profile_{fac}.png"))

print("Exported tables and plots:")
for k, v in out.items():
    print("-", k, ":", v)
print("- plots_dir:", str(plots_dir))


## Reporting Guidance
- Report hypothesis verdicts and factor rulebook ranking in main text.
- Add threshold robustness table and factor-by-threshold heatmap to appendix.
- Use method-wise factor heatmap to discuss transferability.
